In [16]:
from statsbombpy import sb
import pandas as pd
import numpy as np

competitions = sb.competitions()
competitions

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,None,None,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
70,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2024-02-12T14:45:05.702250,2021-06-13T16:17:31.694,None,2024-02-12T14:45:05.702250
71,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2025-07-28T14:19:20.467348,2025-07-29T16:03:07.355174,2025-07-29T16:03:07.355174,2025-07-28T14:19:20.467348
72,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2024-02-13T13:27:17.178263,2024-02-13T13:30:52.820588,2024-02-13T13:30:52.820588,2024-02-13T13:27:17.178263
73,72,107,International,Women's World Cup,female,False,True,2023,2025-07-14T10:07:06.620906,2025-07-14T10:10:27.224586,2025-07-14T10:10:27.224586,2025-07-14T10:07:06.620906


In [17]:
competitions[(competitions["season_name"] == "2023/2024") | (competitions["season_name"] == "2024")]

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
21,223,282,South America,Copa America,male,False,True,2024,2024-11-04T16:15:34.886972,None,None,2024-11-04T16:15:34.886972
68,55,282,Europe,UEFA Euro,male,False,True,2024,2024-09-28T16:51:20.698794,2025-03-24T14:12:30.785094,2025-03-24T14:12:30.785094,2024-09-28T16:51:20.698794


We will use the **2024 Copa America and UEFA Euro** competitions to find clusters of playstyle.

In [18]:
matchesEuro = sb.matches(competition_id=55, season_id=282)
matchesCopaAmerica = sb.matches(competition_id=223, season_id=282)

In [19]:
def circular_mean(angles):
    return np.arctan2(
        np.mean(np.sin(angles)),
        np.mean(np.cos(angles))
    )

def avg_angle(df):
    if len(df) == 0:
        return np.nan
    return circular_mean(df["pass_angle"])

def extract_features(events, team_name):
    features = {}

    # location extraction
    events = events[events["location"].notnull()].copy()
    events["x"] = events["location"].apply(lambda loc: loc[0])

    # split team vs opponent
    team_events = events[events["team"] == team_name]
    opp_events = events[events["team"] != team_name]

    passes = team_events[team_events["type"] == "Pass"]
    shots = team_events[team_events["type"] == "Shot"]
    carries = team_events[team_events["type"] == "Carry"]
    pressures = team_events[team_events["type"] == "Pressure"]
    deep = team_events[team_events["x"] < 40]
    mid = team_events[(team_events["x"] >= 40) & (team_events["x"] < 80)]
    final = team_events[team_events["x"] >= 80]


    # defensive actions
    def_actions = team_events[
        team_events["type"].isin([
            "Pressure", 
            "Duel", 
            "Interception", 
            "Foul Committed"
        ])
    ]

    # opponent passes (for PPDA)
    opp_passes = opp_events[opp_events["type"] == "Pass"]

    # Apply pressing zone filter (x < 80)
    def_actions = def_actions[def_actions["x"] < 80]
    opp_passes = opp_passes[opp_passes["x"] < 80]


    features["passes"] = len(passes)
    features["shots"] = len(shots)
    features["pressures"] = len(pressures)
    features["carries"] = len(carries)
    
    features["xg"] = (
        shots["shot_statsbomb_xg"].sum()
        if "shot_statsbomb_xg" in shots.columns else 0
    )

    features["average_pass_length"] = (
        passes["pass_length"].mean() if len(passes) > 0 else 0
    )

    features["ppda"] = (
        len(opp_passes) / len(def_actions)
        if len(def_actions) > 0 else np.nan
    )

    features["avg_pass_angle_deep"] = avg_angle(deep[deep["type"] == "Pass"])
    features["avg_pass_angle_mid"] = avg_angle(mid[mid["type"] == "Pass"])
    features["avg_pass_angle_final"] = avg_angle(final[final["type"] == "Pass"])

    return features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm

rows = []

matches = pd.concat([matchesCopaAmerica, matchesEuro], ignore_index=True)

for match_id in tqdm(matches.match_id):

    events = sb.events(match_id=match_id)

    teams = events["team"].dropna().unique()

    for team in teams:

        features = extract_features(events, team)

        features["team"] = team
        features["match_id"] = match_id

        rows.append(features)

df = pd.DataFrame(rows)

df.head()

100%|██████████| 83/83 [01:03<00:00,  1.30it/s]


,passes,shots,pressures,carries,xg,average_pass_length,ppda,avg_pass_angle_deep,avg_pass_angle_mid,avg_pass_angle_final,team,match_id
0,537,11,155,413,1.082702,20.786734,3.128655,-0.042155,-0.046114,-0.023094,Argentina,3943077
1,659,19,161,527,0.988178,21.512906,2.338624,-0.034995,-0.128787,1.816389,Colombia,3943077
2,378,19,148,287,6.205310,21.140958,2.108844,-0.107044,0.033943,-0.629554,Canada,3943076
3,365,16,178,290,4.650427,23.869265,1.385417,-0.124896,0.032662,0.152638,Uruguay,3943076
4,460,11,154,364,0.754564,21.812680,1.252688,-0.147674,-0.093207,-0.566419,Uruguay,3942852


In [21]:
df.drop(columns=["match_id"], inplace=True)
df.to_csv(r"../data/matchwise_stats.csv", index=False)

In [22]:
team_stats = df.groupby("team").mean(numeric_only=True).reset_index()

In [23]:
team_stats

,team,passes,shots,pressures,carries,xg,average_pass_length,ppda,avg_pass_angle_deep,avg_pass_angle_mid,avg_pass_angle_final
0,Albania,392.666667,10.666667,146.666667,322.333333,0.727870,21.152318,3.315112,-0.079853,-0.161259,1.125253
1,Argentina,559.666667,14.666667,133.666667,456.666667,2.673967,20.898589,2.424673,0.003169,-0.017089,-1.030759
2,Austria,517.500000,12.500000,180.500000,415.500000,1.528953,20.446254,2.142054,0.142069,0.010456,-0.161990
3,Belgium,551.750000,13.250000,109.750000,471.750000,1.094532,20.983358,3.320037,-0.195753,-0.129669,0.369445
4,Bolivia,408.000000,5.666667,164.000000,315.333333,0.329439,21.921076,2.087439,-0.058614,-0.425858,-0.465007
5,Brazil,530.000000,13.750000,137.000000,438.750000,2.371144,20.577336,2.026312,0.150315,-0.169108,-1.144768
6,Canada,412.000000,12.333333,161.333333,317.666667,2.586989,21.194503,2.279355,0.027886,-0.112432,-0.386921
7,Chile,439.000000,7.333333,154.333333,348.333333,0.536587,23.249089,2.314058,-0.075030,-0.091201,0.189099
8,Colombia,492.000000,13.166667,138.666667,375.166667,1.561297,22.336315,2.272290,-0.010857,0.095794,0.219009
9,Costa Rica,300.333333,3.000000,173.333333,224.666667,0.158028,24.756273,2.371913,-0.082550,0.269005,0.531436


In [24]:
team_stats.to_csv(r'../data/team_stats_raw.csv', index=False)